# Analisis Durasi Perjalanan Bike Share San Francisco

**Tujuan:** menganalisis `duration_sec` dari tabel publik BigQuery `bigquery-public-data.san_francisco_bikeshare.bikeshare_trips` (maksimal 3.000 baris), melakukan deteksi dan handling outlier, lalu mengekspor data bersih untuk digunakan oleh API FastAPI.

> Jalankan notebook ini di Google Colab. Autentikasi dan penggunaan kuota/query BigQuery dilakukan melalui akun Google Cloud milik sendiri. Isi `project_id` dengan Project ID GCP milikmu, bukan `bigquery-public-data`.

## 1. Persiapan dan koneksi BigQuery

In [ ]:
!pip -q install google-cloud-bigquery pandas-gbq db-dtypes seaborn matplotlib

In [ ]:
from google.colab import auth
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

auth.authenticate_user()
print("Authenticated")

project_id = "GANTI_DENGAN_GCP_PROJECT_ID_KAMU"
client = bigquery.Client(project=project_id)

query = '''
SELECT duration_sec
FROM `bigquery-public-data.san_francisco_bikeshare.bikeshare_trips`
WHERE duration_sec IS NOT NULL
LIMIT 3000
'''

df = client.query(query).to_dataframe()
print("Ukuran data:", df.shape)
display(df.head())
print(df.info())

## 2. Data understanding dan cleaning awal

Kolom yang dianalisis hanya `duration_sec`. Nilai kosong dihapus dan nilai durasi tidak positif dikeluarkan karena durasi perjalanan harus lebih besar dari nol.

In [ ]:
df = df[['duration_sec']].copy()
df['duration_sec'] = pd.to_numeric(df['duration_sec'], errors='coerce')
df = df.dropna(subset=['duration_sec'])
df = df[df['duration_sec'] > 0].reset_index(drop=True)

# ID stabil untuk kebutuhan endpoint DELETE.
df.insert(0, 'record_id', range(1, len(df) + 1))

print("Jumlah data valid:", len(df))
print("Missing value:")
display(df.isna().sum().to_frame('missing_count'))
display(df.describe())

## 3. Statistik deskriptif sebelum handling outlier

In [ ]:
def descriptive_stats(series):
    s = series.dropna()
    mode_values = s.mode().tolist()
    return pd.Series({
        'count': s.count(),
        'mean_sec': s.mean(),
        'median_sec': s.median(),
        'mode_sec': mode_values,
        'std_sec': s.std(),
        'variance_sec2': s.var(),
        'range_sec': s.max() - s.min(),
        'min_sec': s.min(),
        'max_sec': s.max(),
        'skewness': s.skew(),
        'kurtosis_excess': s.kurt()
    })

stats_before = descriptive_stats(df['duration_sec'])
display(stats_before.to_frame('before_outlier_handling'))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df['duration_sec'], bins=40, kde=True, ax=axes[0])
axes[0].set_title('Distribusi Durasi Sebelum Handling')
axes[0].set_xlabel('Duration (seconds)')
sns.boxplot(x=df['duration_sec'], ax=axes[1])
axes[1].set_title('Boxplot Sebelum Handling')
plt.tight_layout()
plt.show()

## 4. Deteksi outlier dengan metode IQR

Metode IQR dipilih karena tidak mengasumsikan distribusi normal dan relatif tahan terhadap nilai ekstrem.

- IQR = Q3 − Q1
- Batas bawah = Q1 − 1,5 × IQR
- Batas atas = Q3 + 1,5 × IQR
- Nilai di luar kedua batas tersebut dikategorikan sebagai outlier.

In [ ]:
q1 = df['duration_sec'].quantile(0.25)
q3 = df['duration_sec'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outlier_mask = (df['duration_sec'] < lower_bound) | (df['duration_sec'] > upper_bound)
outliers = df.loc[outlier_mask].copy()
df_clean = df.loc[~outlier_mask].copy().reset_index(drop=True)

total_before = len(df)
n_outlier = len(outliers)
outlier_pct = (n_outlier / total_before * 100) if total_before else 0

print(f"Q1             : {q1:,.2f} detik")
print(f"Q3             : {q3:,.2f} detik")
print(f"IQR            : {iqr:,.2f} detik")
print(f"Batas bawah    : {lower_bound:,.2f} detik")
print(f"Batas atas     : {upper_bound:,.2f} detik")
print(f"Jumlah awal    : {total_before:,}")
print(f"Jumlah outlier : {n_outlier:,}")
print(f"Persentase     : {outlier_pct:.2f}%")
print(f"Data tersisa   : {len(df_clean):,}")

display(outliers.head(10))

## 5. Visualisasi setelah handling outlier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df_clean['duration_sec'], bins=40, kde=True, ax=axes[0])
axes[0].set_title('Distribusi Durasi Setelah Handling')
axes[0].set_xlabel('Duration (seconds)')
sns.boxplot(x=df_clean['duration_sec'], ax=axes[1])
axes[1].set_title('Boxplot Setelah Handling')
plt.tight_layout()
plt.show()

## 6. Perbandingan pemusatan data sebelum dan sesudah

Mean sensitif terhadap nilai ekstrem, sedangkan median lebih robust. Modus menunjukkan durasi yang paling sering muncul; jika ada beberapa modus, semuanya ditampilkan.

In [ ]:
stats_after = descriptive_stats(df_clean['duration_sec'])
comparison = pd.DataFrame({
    'before': stats_before[['mean_sec', 'median_sec', 'mode_sec']],
    'after': stats_after[['mean_sec', 'median_sec', 'mode_sec']]
})
display(comparison)

print("Interpretasi otomatis:")
print(f"- Mean: {stats_before['mean_sec']:.2f} -> {stats_after['mean_sec']:.2f} detik")
print(f"- Median: {stats_before['median_sec']:.2f} -> {stats_after['median_sec']:.2f} detik")
print(f"- Jumlah data berkurang {n_outlier:,} baris ({outlier_pct:.2f}%).")
print("Bandingkan perubahan mean dan median untuk melihat seberapa besar pengaruh nilai ekstrem.")

## 7. Simpan dataset hasil handling outlier

File `bike_duration_clean.csv` akan dipakai oleh API. `record_id` dipertahankan sebagai identitas unik setiap baris.

In [ ]:
csv_path = 'bike_duration_clean.csv'
df_clean.to_csv(csv_path, index=False)
print(f"Berhasil menyimpan {csv_path}")
print("Jumlah record:", len(df_clean))

from google.colab import files
files.download(csv_path)

## 8. Kesimpulan

Lengkapi kesimpulan berdasarkan output aktual notebook:

1. **Central tendency:** laporkan mean, median, dan modus sebelum dan sesudah.
2. **Dispersi:** bandingkan standar deviasi, variance, dan range.
3. **Bentuk distribusi:** skewness positif menunjukkan ekor kanan lebih panjang; negatif menunjukkan ekor kiri lebih panjang. Kurtosis pada Pandas adalah *excess kurtosis*: nilai positif lebih berat ekornya daripada distribusi normal, nilai negatif lebih ringan.
4. **Outlier:** laporkan batas IQR, jumlah outlier, dan persentasenya.
5. **Dampak handling:** jelaskan perubahan mean/median/modus. Jangan menyimpulkan outlier sebagai kesalahan data secara otomatis; durasi ekstrem bisa merupakan perjalanan nyata atau anomali pencatatan.